# 00 Setup และสเปกเครื่อง

โปรเจกต์วิเคราะห์ข้อมูลผู้โดยสาร BART (2011-2023) ประมาณ 112 ล้านแถว

เลือกทำบนเครื่องตัวเองแทน PySpark เพราะข้อมูล ~3 GB ใส่ใน RAM 32 GB ได้หมด ไม่จำเป็นต้องใช้ cluster
ใช้ DuckDB / Polars ที่ทำงานขนานได้ทุก core และใช้ GPU ตอนเทรนโมเดล

notebook นี้เช็ค environment, ดูสเปกเครื่อง, ทดสอบ GPU และดูข้อมูลดิบคร่าวๆ

## โหลดโมดูลของโปรเจกต์

In [1]:
import importlib
import platform
import subprocess
import sys
from pathlib import Path

if Path.cwd().name == "notebooks":
    ROOT = Path.cwd().parent
else:
    ROOT = Path.cwd()
sys.path.append(str(ROOT / "src"))

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import psutil
import xgboost as xgb

from bart import viz
from bart.config import RAW, duck
from bart.perf import stage

viz.notebook_mode()

## สเปกเครื่อง

ความเร็วแรม (DDR5-5200) ต้องใช้ sudo ถึงจะอ่านได้ เลยดูจาก `docs/machine_spec.md`

In [2]:
def read_file(path):
    try:
        return Path(path).read_text().strip()
    except OSError:
        return ""


def parse_cpus(cpu_range):
    # "0-11" -> [0, 1, ..., 11], "12-19,22" -> [12, ..., 19, 22]
    cpus = []
    for part in cpu_range.split(","):
        if part == "":
            continue
        if "-" in part:
            start, end = part.split("-")
        else:
            start = part
            end = part
        for cpu in range(int(start), int(end) + 1):
            cpus.append(cpu)
    return cpus


def cpu_model():
    for line in read_file("/proc/cpuinfo").splitlines():
        if line.startswith("model name"):
            return line.split(":", 1)[1].strip()
    return "unknown"


def os_name():
    for line in read_file("/etc/os-release").splitlines():
        if line.startswith("PRETTY_NAME="):
            return line.split("=", 1)[1].strip('"')
    return "unknown"


def power_profile():
    try:
        result = subprocess.run(["powerprofilesctl", "get"], capture_output=True, text=True)
    except OSError:
        return "unknown"
    return result.stdout.strip()


def max_clock_ghz(freqs, cpus):
    highest_mhz = 0.0
    for cpu in cpus:
        if freqs[cpu].max > highest_mhz:
            highest_mhz = freqs[cpu].max
    return highest_mhz / 1000


def gpu_info():
    fields = "name,memory.total,driver_version,compute_cap,pcie.link.gen.max,pcie.link.width.max"
    command = ["nvidia-smi", f"--query-gpu={fields}", "--format=csv,noheader,nounits"]
    result = subprocess.run(command, capture_output=True, text=True)
    name, memory_mb, driver, compute, pcie_gen, pcie_width = result.stdout.strip().split(", ")
    return {
        "name": name,
        "memory_gb": int(memory_mb) / 1024,
        "driver": driver,
        "compute": compute,
        "pcie": f"Gen{pcie_gen} x{pcie_width}",
    }

In [3]:
p_cpus = parse_cpus(read_file("/sys/devices/cpu_core/cpus"))
e_cpus = parse_cpus(read_file("/sys/devices/cpu_atom/cpus"))

# psutil.cpu_freq() averages the max clock over all threads (= 4.8 GHz here),
# so take the highest value per core type instead
freqs = psutil.cpu_freq(percpu=True)
p_max_ghz = max_clock_ghz(freqs, p_cpus)
e_max_ghz = max_clock_ghz(freqs, e_cpus)

mem = psutil.virtual_memory()
disk = psutil.disk_usage(str(ROOT))
gpu = gpu_info()

vendor = read_file("/sys/class/dmi/id/sys_vendor")
product = read_file("/sys/class/dmi/id/product_name")
ram_total_gib = mem.total / 2**30
ram_free_gib = mem.available / 2**30

spec = {
    "Machine": f"{vendor} {product}",
    "OS": f"{os_name()} (kernel {platform.release()})",
    "CPU": cpu_model(),
    "Cores / threads": f"{psutil.cpu_count(logical=False)} / {psutil.cpu_count()}",
    "P-cores": f"{len(p_cpus) // 2} cores ({len(p_cpus)} threads)",
    "E-cores": f"{len(e_cpus)} cores",
    "Max clock (P-core)": f"{p_max_ghz:.1f} GHz",
    "Max clock (E-core)": f"{e_max_ghz:.1f} GHz",
    "RAM": f"{ram_total_gib:.1f} GiB DDR5-5200 ({ram_free_gib:.1f} GiB free)",
    "GPU": gpu["name"],
    "VRAM": f"{gpu['memory_gb']:.0f} GB",
    "CUDA compute / PCIe": f"{gpu['compute']} / {gpu['pcie']}",
    "GPU driver": gpu["driver"],
    "Disk": f"{disk.total / 1e9:.0f} GB NVMe ({disk.free / 1e9:.0f} GB free)",
    "Power profile": power_profile(),
}

pd.Series(spec, name="value").to_frame()

,value
Machine,"Micro-Star International Co., Ltd. Katana 15 B..."
OS,Ubuntu 26.04.1 LTS (kernel 7.0.0-38-generic)
CPU,13th Gen Intel(R) Core(TM) i9-13900H
Cores / threads,14 / 20
P-cores,6 cores (12 threads)
E-cores,8 cores
Max clock (P-core),5.4 GHz
Max clock (E-core),4.1 GHz
RAM,30.6 GiB DDR5-5200 (25.4 GiB free)
GPU,NVIDIA GeForce RTX 4060 Laptop GPU


## ไลบรารีและทดสอบ GPU

ลองเทรน XGBoost บน GPU ด้วยข้อมูลสุ่มดูว่าใช้ได้ไหม

In [4]:
libraries = [
    "duckdb", "polars", "pyarrow", "psutil", "numpy", "pandas",
    "sklearn", "xgboost", "shap", "optuna", "plotly", "holidays",
]

versions = {}
for name in libraries:
    module = importlib.import_module(name)
    versions[name] = module.__version__

pd.Series(versions, name="version").to_frame()

,version
duckdb,1.5.6
polars,2.0.0
pyarrow,26.0.0
psutil,7.2.2
numpy,2.5.4
pandas,3.0.6
sklearn,1.9.1
xgboost,3.4.1
shap,0.53.0
optuna,5.0.0


In [5]:
print("XGBoost built with CUDA:", xgb.build_info()["USE_CUDA"])

rng = np.random.default_rng(42)
X = rng.random((200_000, 20), dtype=np.float32)
weights = rng.random(20, dtype=np.float32)
y = X @ weights

params = {"device": "cuda", "tree_method": "hist", "max_depth": 8}

with stage("00.gpu_test", gpu=True):
    train_matrix = xgb.QuantileDMatrix(X, y)
    model = xgb.train(params, train_matrix, num_boost_round=200)

XGBoost built with CUDA: True


[perf] 00.gpu_test: 2.68s | 2.2 cores | peak RAM 0.594 GB | CPU 12.1% | GPU mem 1120 MB


## ข้อมูลดิบ

CSV ปีละไฟล์ มี 5 คอลัมน์ Date, Hour, Origin Station, Destination Station, Trip Count
ใช้ DuckDB อ่าน CSV ตรงๆ ไปก่อน แปลงเป็น Parquet ใน notebook 01

In [6]:
csv_files = sorted(RAW.glob("*.csv"))

file_rows = []
for file in csv_files:
    file_rows.append({"file": file.name, "size_mb": round(file.stat().st_size / 1e6, 1)})
files = pd.DataFrame(file_rows)

total_gb = files["size_mb"].sum() / 1000
print(f"{len(files)} files, {total_gb:.2f} GB")
files

13 files, 2.92 GB


,file,size_mb
0,date-hour-soo-dest-2011.csv,245.2
1,date-hour-soo-dest-2012.csv,254.1
2,date-hour-soo-dest-2013.csv,249.9
3,date-hour-soo-dest-2014.csv,257.3
4,date-hour-soo-dest-2015.csv,262.1
5,date-hour-soo-dest-2016.csv,259.5
6,date-hour-soo-dest-2017.csv,259.4
7,date-hour-soo-dest-2018.csv,265.8
8,date-hour-soo-dest-2019.csv,262.9
9,date-hour-soo-dest-2020.csv,150.2


In [7]:
con = duck()
csv = f"read_csv('{RAW}/*.csv', header=true)"

con.sql(f"DESCRIBE SELECT * FROM {csv}").df()

,column_name,column_type,null,key,default,extra
0,Date,DATE,YES,None,None,None
1,Hour,BIGINT,YES,None,None,None
2,Origin Station,VARCHAR,YES,None,None,None
3,Destination Station,VARCHAR,YES,None,None,None
4,Trip Count,BIGINT,YES,None,None,None


In [8]:
con.sql(f"SELECT * FROM {csv} LIMIT 5").df()

,Date,Hour,Origin Station,Destination Station,Trip Count
0,2011-01-01,0,12TH,12TH,1
1,2011-01-01,0,12TH,16TH,1
2,2011-01-01,0,12TH,24TH,3
3,2011-01-01,0,12TH,ASHB,2
4,2011-01-01,0,12TH,BAYF,5


In [9]:
query = f"""
    SELECT
        year("Date")           AS year,
        count(*)               AS rows,
        sum("Trip Count")      AS trips,
        count(DISTINCT "Date") AS days
    FROM {csv}
    GROUP BY year
    ORDER BY year
"""

with stage("00.raw_csv_profile"):
    by_year = con.sql(query).df()

print(f"rows: {by_year['rows'].sum():,}  trips: {by_year['trips'].sum():,}")
by_year

[perf] 00.raw_csv_profile: 1.66s | 17.63 cores | peak RAM 1.917 GB | CPU 90.8%
rows: 112,536,132  trips: 1,194,695,771.0


,year,rows,trips,days
0,2011,9428124,106423154.0,363
1,2012,9771799,115445456.0,365
2,2013,9606163,116968634.0,363
3,2014,9891760,124052914.0,365
4,2015,10074967,127760948.0,365
5,2016,9973949,128208307.0,366
6,2017,9974158,122879064.0,365
7,2018,10229648,120518643.0,365
8,2019,10114285,118609605.0,365
9,2020,5811863,31426279.0,362


ตัวเลขนี้นับจาก CSV ดิบ ยังรวมแถวซ้ำ 16,078 แถว (ช่วงตี 0-2 ของวันปีใหม่ ที่ติดมาในไฟล์ปีก่อนหน้า ดูใน notebook 01)
แถวซ้ำถูกนับเข้าปีตามวันที่ ปี 2012-2018 จึงมี rows และ trips เกินจริงเล็กน้อย ส่วน days ไม่เปลี่ยน

In [10]:
colors = []
for year in by_year["year"]:
    if year >= 2020:
        colors.append(viz.CONTRAST)
    else:
        colors.append(viz.ACCENT)

bars = go.Bar(
    x=by_year["year"],
    y=by_year["rows"],
    marker_color=colors,
    customdata=by_year[["trips", "days"]],
    hovertemplate="%{x}<br>%{y:,} rows<br>%{customdata[0]:,} trips in %{customdata[1]} days<extra></extra>",
)

fig = go.Figure(bars)
fig.update_layout(hovermode="closest", showlegend=False)
fig.update_yaxes(tickformat="~s")

viz.titled(fig, "Rows per year", "OD-hour rows with at least one trip. Orange = 2020 onwards, 2023 = Jan to May 2")
viz.note(fig, "Source: BART hourly origin-destination data")
viz.save(fig, "00_rows_per_year")
fig

## สรุปเครื่องมือที่ใช้

- DuckDB: แปลง CSV เป็น Parquet และ query หนักๆ ใช้ได้ 20 threads
- Parquet (zstd): เก็บข้อมูล ไฟล์เล็กลง อ่านเร็วขึ้น อ่านเฉพาะคอลัมน์ที่ต้องใช้
- Polars (lazy): ทำ feature สำหรับโมเดล
- XGBoost (GPU): โมเดลพยากรณ์ เพราะข้อมูลหลายล้านแถว
- scikit-learn: clustering สถานี ข้อมูลน้อย ใช้ CPU พอ
- psutil: จับเวลาและการใช้ RAM/CPU ของแต่ละขั้นตอน

ต่อไป: 01_ingest